In [ ]:
%%sql -r dataframe_2
USE WAREHOUSE COMPUTE_WH;

USE DATABASE LEAGUE_RECORDS;

USE SCHEMA SILVER;


In [ ]:
from typing import Annotated

from snowflake.snowpark import DataFrame as SnowflakeDataFrame
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pydantic import (
    BaseModel,
    Field,
    field_validator,
    model_validator,
)
import statsmodels.formula.api as smf
from statsmodels.regression.linear_model import RegressionResults

from scipy.stats import norm

In [ ]:
import sys
import os

workspace_root = os.path.dirname(
    os.path.dirname(os.getcwd())
)
sys.path.insert(0, workspace_root)

from analysis.schemas import AnalysisQueryParameters

## Base Dataset: `SILVER.PLAYER_INTERVALS_SILVER`


In [ ]:
%%sql -r base_dataset
SELECT *
FROM SILVER.PLAYER_INTERVAL_SILVER
;

In [ ]:
class BivariateQueryParams(AnalysisQueryParameters):
    x_var: str
    y_var: str
    x_lim: tuple
    y_lim: tuple

    @field_validator('x_var', 'y_var', mode='before')
    @classmethod
    def normalize_to_uppercase(cls, v: str) -> str:
        return v.upper()

    @property
    def plot_title(self) -> str:
        return f"Bivariate relationship between {self.x_var} VS. {self.y_var}"

    def query(self, df: SnowflakeDataFrame):
        return (df
            .to_pandas()
            [[self.x_var, self.y_var]]
        )

    def __str__(self) -> str:
        return (
            f"Query --> Predicting {self.y_var} (limit: {self.y_lim[0]} → {self.y_lim[1]}) "
            f"from {self.x_var} (limit: {self.x_lim[0]} → {self.x_lim[1]})"
        )

In [ ]:
print(BivariateQueryParams(
    x_var="KILLS",
    y_var="GOLDS",
    x_lim=(0, 30),
    y_lim=(0, 20000)
))

In [ ]:
def sample_xy(
    df: SnowflakeDataFrame,
    x_var: str,
    y_var: str,
    sample_size: int = 100,
    random_state: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    sample_data = df.sample(sample_size, replace=True, random_state=random_state)

    x_arr = sample_data[x_var].to_numpy()
    y_arr = sample_data[y_var].to_numpy()

    return (x_arr, y_arr)

In [ ]:
def visualize_regplot(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
    params: BivariateQueryParams,
    **kwargs
) -> None:
    fig, ax = plt.subplots(figsize=(10,6), dpi=150)
    sns.regplot(
        x=x_arr,
        y=y_arr,
        ax=ax,
        **kwargs
    )

    ax.set_title(params.plot_title)
    ax.set_xlabel(params.x_var)
    ax.set_ylabel(params.y_var)
    ax.set_xlim(params.x_lim)
    ax.set_ylim(params.y_lim)
    
    plt.show()
    plt.close(fig)

In [ ]:
def calc_least_square_line(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
    round_decimal: int | None = None
) -> tuple[float, float]:
    x_ = np.mean(x_arr)
    y_ = np.mean(y_arr)
    xy_ = np.mean(x_arr * y_arr)
    x_sqr_ = np.mean(x_arr ** 2)

    m = (xy_ - (x_ * y_)) / (x_sqr_ - (x_**2))
    b = y_ - (m * x_)

    if round_decimal:
        m = np.round(m, round_decimal)
        b = np.round(b, round_decimal)

    print(f"Line of best fit formula: y = {m:.2f}x + {b:.2f}")

    return (m, b)

In [ ]:
def bivariate_rls(
    data: SnowflakeDataFrame,
    x_var: str,
    y_var: str,
    x_lim: tuple,
    y_lim: tuple,
) -> None:
    params = BivariateQueryParams(
        x_var=x_var,
        y_var=y_var,
        x_lim=x_lim,
        y_lim=y_lim
    )
    
    x_arr, y_arr = sample_xy(
        params.query(data),
        params.x_var,
        params.y_var
    )
    print(x_arr, y_arr)

    calc_least_square_line(x_arr, y_arr)

    visualize_regplot(x_arr, y_arr, params)


In [ ]:
bivariate_rls(
    data=base_dataset,
    x_var="KILLS",
    y_var="TOTAL_GOLD",
    x_lim=(0, 30),
    y_lim=(0, 20000)
)

## Using Statsmodel

In [ ]:
def sample_dataset(
    df: SnowflakeDataFrame,
    sample_size: int = 100,
    seed: int = 42
) -> pd.DataFrame:
    return (df
        .to_pandas()
        .sample(sample_size, replace=True, random_state=seed)
    )

In [ ]:
def sqrt_adjust_var(
    df: pd.DataFrame, 
    cols_to_adj: list[str]
) -> pd.DataFrame:
    has_invalid_cols = any(col for col in cols_to_adj if col not in df.columns)
    if has_invalid_cols:
        raise ValueError(f"Invalid column input! Got {cols_to_adj}. Make sure it is a valid column name within the dataframe.")
    
    assign_mapping = {
        f"SQRT_{col.upper()}": np.sqrt(df[col])
        for col in cols_to_adj
    }
    
    return df.assign(**assign_mapping)

In [ ]:
def build_linear_reg(
    data: pd.DataFrame,
    explanatory_var: str,
    response_var: str
) -> RegressionResults:
    model = smf.ols(
        formula=f"{response_var} ~ {explanatory_var}", 
        data=data
    ).fit()
    print(model.summary())
    
    return model

In [ ]:
def build_residual_plot(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
) -> None: 
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    sns.residplot(x=x_arr, y=y_arr, ax=ax)
    
    ax.set_title("Residual Plot")
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Residuals")
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    
    plt.show()
    plt.close(fig)

In [ ]:
def run(
    x_var: str, 
    y_var: str,
    predict_for: int,
    sample_size: int,
    sqrt_x: bool = False,
    sqrt_y: bool = False,
) -> None:
    sample_data = sample_dataset(
        base_dataset, 
        sample_size=sample_size
    )[[x_var, y_var]]
    cols_to_adj = [
        *([x_var] if sqrt_x else []),
        *([y_var] if sqrt_y else [])
    ]
    if cols_to_adj:
        sample_data = sqrt_adjust_var(sample_data, cols_to_adj)

    effective_x = f"SQRT_{x_var}" if sqrt_x else x_var
    effective_y = f"SQRT_{y_var}" if sqrt_y else y_var
        
    model = build_linear_reg(
        sample_data,
        explanatory_var=effective_x,
        response_var=effective_y
    )
    predict_value = np.sqrt(predict_for) if sqrt_x else predict_for
    result = model.predict(pd.DataFrame({effective_x: [predict_value]})).to_numpy()

    x_arr = sample_data[effective_x].to_numpy()
    y_arr = sample_data[effective_y].to_numpy()
    build_residual_plot(x_arr, y_arr)
    
    print(f"Prediction for {y_var} at {predict_for} {x_var} --> {result[0]:.2f} {y_var}")



In [ ]:
run(
    "KILLS", 
    "TOTAL_GOLD",
    predict_for=10,
    sample_size=1000,
)